# 05. Huấn luyện

Notebook này ghép môi trường, Experience Replay và agent thành vòng huấn luyện DQN, đánh giá định kỳ và lưu checkpoint tốt nhất.

Nội dung:

1. Nạp các module.
2. Cấu hình.
3. Hàm `train`.
4. Chạy huấn luyện.
5. Vẽ đường học.

Mã giả và lý do chọn từng tham số nằm ở mục 5 và 6 của `Implementation_Notes.md`.

## 1. Nạp các module

`%run` chạy các notebook trước trong cùng kernel, kể cả phần kiểm tra nhanh của chúng (mất vài giây). `04_agent` tự nạp `02_q_network`.

In [ ]:
%run ./01_environment.ipynb
%run ./03_replay_buffer.ipynb
%run ./04_agent.ipynb

## 2. Cấu hình

Các giá trị lấy từ mục 5 của `Implementation_Notes.md`. Chúng được chọn bằng lập luận và **chưa được kiểm chứng bằng huấn luyện**.

| Tham số | Giá trị | Ghi chú |
|---|---|---|
| `total_frames` | 1.000.000 | Mức tối thiểu để giá trị lan đủ xa với γ = 0,99 |
| `gamma` | 0,99 | |
| `learning_rate` | 0,0001 | Adam |
| `buffer_capacity` | 100.000 | |
| `learning_starts` | 10.000 | Bắt đầu học khi Experience Replay có đủ chừng này transition |
| `batch_size` | 64 | |
| `update_every` | 4 | Cập nhật mạng mỗi 4 khung hình |
| `target_sync_every` | 1.000 | Tính theo **số lần cập nhật**, tức 4.000 khung hình |
| ε | 1,0 về 0,01 trong 100.000 khung hình | |
| Đánh giá | Mỗi 25.000 khung hình, 20 ván, `score_limit` 200 | Môi trường riêng, seed cố định khác seed huấn luyện |

Checkpoint được lưu vào thư mục `checkpoints/` cạnh notebook.

In [ ]:
import os
import json
import time
import numpy as np
import keras

CONFIG = dict(
    total_frames=1_000_000,
    gamma=0.99,
    learning_rate=1e-4,
    hidden_units=256,
    buffer_capacity=100_000,
    learning_starts=10_000,
    batch_size=64,
    update_every=4,
    target_sync_every=1_000,
    epsilon_start=1.0,
    epsilon_end=0.01,
    epsilon_decay_frames=100_000,
    eval_every=25_000,
    eval_episodes=20,
    eval_score_limit=200,
    train_seed=0,
    eval_seed=12345,
    checkpoint_dir="checkpoints",
)

## 3. Hàm `train`

Mỗi khung hình gồm năm việc:

1. Chọn hành động bằng ε-greedy.
2. Chơi một khung hình và lưu transition, kèm cờ `terminated`, vào Experience Replay. Nếu ván kết thúc, gọi `reset()`.
3. Cứ 4 khung hình, lấy ngẫu nhiên 64 transition và cập nhật mạng một bước theo Target của Double DQN.
4. Cứ 1.000 lần cập nhật, chép trọng số mạng chính sang Target Network.
5. Cứ 25.000 khung hình, đánh giá với ε = 0 trên một môi trường riêng và lưu trọng số nếu điểm trung bình tốt nhất từ trước tới giờ.

Môi trường huấn luyện không đặt `score_limit`, nên ván chỉ kết thúc khi chim va chạm. Lớp `FlappyEnv` báo lỗi nếu lỡ gọi `step()` sau va chạm, nên Experience Replay không nhận transition của con chim đã chết.

Mỗi lần đánh giá in ra một dòng gồm: số khung hình, số ván đã chơi, ε, điểm trung bình các ván huấn luyện gần đây, loss và trung bình max Q của các lần cập nhật gần đây, cùng điểm đánh giá. Khi kết thúc, hàm lưu `last.weights.h5` và `history.json`.

In [ ]:
def _mean_or_nan(values):
    return float(np.mean(values)) if values else float("nan")


def train(config):
    keras.utils.set_random_seed(config["train_seed"])
    os.makedirs(config["checkpoint_dir"], exist_ok=True)
    best_path = os.path.join(config["checkpoint_dir"], "best.weights.h5")

    env = FlappyEnv()
    buffer = ReplayBuffer(config["buffer_capacity"], OBS_DIM, seed=config["train_seed"])
    agent = DQNAgent(OBS_DIM, N_ACTIONS, config["gamma"], config["learning_rate"],
                     config["hidden_units"], seed=config["train_seed"])

    history = []
    best_eval_mean = -np.inf
    n_updates = 0
    n_episodes = 0
    recent_scores, recent_losses, recent_q = [], [], []
    start_time = time.time()

    state = env.reset(seed=config["train_seed"])
    try:
        for frame in range(1, config["total_frames"] + 1):
            # 1. Chọn hành động
            epsilon = linear_epsilon(frame, config["epsilon_start"], config["epsilon_end"],
                                     config["epsilon_decay_frames"])
            action = agent.act(state, epsilon)

            # 2. Chơi một khung hình và lưu transition
            next_state, reward, terminated, truncated, score = env.step(action)
            buffer.add(state, action, reward, next_state, terminated)
            if terminated or truncated:
                recent_scores.append(score)
                n_episodes += 1
                state = env.reset()
            else:
                state = next_state

            # 3 và 4. Cập nhật mạng chính, đồng bộ Target Network
            if len(buffer) >= config["learning_starts"] and frame % config["update_every"] == 0:
                loss, mean_max_q = agent.update(*buffer.sample(config["batch_size"]))
                recent_losses.append(loss)
                recent_q.append(mean_max_q)
                n_updates += 1
                if n_updates % config["target_sync_every"] == 0:
                    agent.sync_target()

            # 5. Đánh giá và lưu checkpoint tốt nhất
            if frame % config["eval_every"] == 0:
                scores = play_episodes(agent.greedy_action, config["eval_episodes"],
                                       config["eval_seed"], config["eval_score_limit"])
                stats = summarize_scores(scores)
                record = dict(frame=frame, episodes=n_episodes, updates=n_updates, epsilon=epsilon,
                              train_score=_mean_or_nan(recent_scores),
                              loss=_mean_or_nan(recent_losses),
                              mean_max_q=_mean_or_nan(recent_q),
                              eval=stats, minutes=(time.time() - start_time) / 60)
                history.append(record)

                is_best = stats["mean"] > best_eval_mean
                if is_best:
                    best_eval_mean = stats["mean"]
                    agent.save(best_path)
                print(f"frame {frame:>9,} | ván {n_episodes:>6} | ε {epsilon:.3f} | "
                      f"điểm train {record['train_score']:6.2f} | loss {record['loss']:.4f} | "
                      f"max Q {record['mean_max_q']:6.2f} | đánh giá {stats['mean']:6.2f} "
                      f"(trung vị {stats['median']:.0f}, {stats['min']} đến {stats['max']})"
                      f"{' *' if is_best else ''} | {record['minutes']:.1f} phút")
                recent_scores, recent_losses, recent_q = [], [], []
    finally:
        env.close()
        agent.save(os.path.join(config["checkpoint_dir"], "last.weights.h5"))
        with open(os.path.join(config["checkpoint_dir"], "history.json"), "w", encoding="utf-8") as f:
            json.dump(history, f, ensure_ascii=False, indent=1)

    return agent, history

## 4. Chạy huấn luyện

Thời gian ước tính trên CPU: khoảng 12 phút cho phần huấn luyện. Khi agent đã giỏi, mỗi lần đánh giá có thể kéo dài tới khoảng 45 giây, nên tổng cộng khoảng 15 đến 45 phút.

Dòng có dấu `*` là lần đánh giá tốt nhất từ trước tới giờ, và trọng số tại đó đã được lưu vào `checkpoints/best.weights.h5`. Có thể dừng giữa chừng bằng nút Interrupt: trọng số cuối và lịch sử vẫn được lưu.

In [ ]:
agent, history = train(CONFIG)

## 5. Đường học

- Hình trái: điểm đánh giá theo số khung hình. Đường học của DQN thường không tăng đều, vì vậy dự án giữ checkpoint tốt nhất thay vì checkpoint cuối.
- Hình phải: trung bình max Q của các batch. Đường nét đứt là 12,9, mức cao nhất mà giá trị thật có thể đạt với γ = 0,99. Q vượt rõ mức này là dấu hiệu đánh giá quá cao hoặc việc học đang phân kỳ.

In [ ]:
import matplotlib.pyplot as plt

frames = [r["frame"] for r in history]
fig, (ax_score, ax_q) = plt.subplots(1, 2, figsize=(12, 4))

ax_score.plot(frames, [r["eval"]["mean"] for r in history], label="trung bình")
ax_score.plot(frames, [r["eval"]["median"] for r in history], label="trung vị")
ax_score.axhline(34.2, color="gray", linestyle=":", label="luật viết tay (34,2)")
ax_score.set_xlabel("khung hình")
ax_score.set_ylabel("số ống vượt được")
ax_score.set_title("Điểm đánh giá (ε = 0)")
ax_score.legend()

ax_q.plot(frames, [r["mean_max_q"] for r in history])
ax_q.axhline(12.9, color="red", linestyle="--", label="giới hạn giá trị thật (12,9)")
ax_q.set_xlabel("khung hình")
ax_q.set_ylabel("trung bình max Q")
ax_q.set_title("Q của các batch huấn luyện")
ax_q.legend()

plt.tight_layout()
plt.show()